# Assignment 5, Question 6: Data Transformation

**Points: 20**

Transform and engineer features from the clinical trial dataset.

## Setup

In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import utilities
from q3_data_utils import load_data, clean_data, transform_types, create_bins, fill_missing

df = load_data('data/clinical_trial_raw.csv')
print(f"Loaded {len(df)} patients")

# Prewritten visualization functions for transformation analysis
def plot_distribution(series, title, figsize=(10, 6)):
    """
    Create a histogram of a numeric series.
    
    Args:
        series: pandas Series with numeric data
        title: Chart title
        figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)
    series.hist(bins=30)
    plt.title(title)
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

def plot_value_counts(series, title, figsize=(10, 6)):
    """
    Create a bar chart of value counts.
    
    Args:
        series: pandas Series with value counts
        title: Chart title
        figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)
    series.plot(kind='bar')
    plt.title(title)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

Loaded 10000 patients


## Part 1: Type Conversions (5 points)

1. Convert 'enrollment_date' to datetime using the `transform_types()` utility
2. Convert categorical columns ('site', 'intervention_group', 'sex') to category dtype
3. Ensure all numeric columns are proper numeric types
4. Display the updated dtypes

In [56]:
# TODO: Type conversions
# 1. Use transform_types() to convert enrollment_date to datetime
# 2. Convert categorical columns ('site', 'intervention_group', 'sex') to category dtype
# 3. Ensure all numeric columns are proper numeric types
# 4. Display the updated dtypes using df.dtypes
print(df.dtypes)
type_map = {'enrollment_date': 'datetime','sex': 'category','site': 'category', 'intervention_group':'category'}
df_typed = transform_types(df, type_map)
#ensure numeric columns
numeric_columns = df_typed.select_dtypes(include = ['number']).columns
print(f" the numeric columns are:\n {numeric_columns}")
display(df_typed.dtypes)

patient_id             object
age                     int64
sex                    object
bmi                   float64
enrollment_date        object
systolic_bp           float64
diastolic_bp          float64
cholesterol_total     float64
cholesterol_hdl       float64
cholesterol_ldl       float64
glucose_fasting       float64
site                   object
intervention_group     object
follow_up_months        int64
adverse_events          int64
outcome_cvd            object
adherence_pct         float64
dropout                object
dtype: object
enrollment_date is present in the data frame
enrollment_date was converted to datetime
sex is present in the data frame
sex was converted to category
site is present in the data frame
site was converted to category
intervention_group is present in the data frame
intervention_group was converted to category
 the numeric columns are:
 Index(['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'cholesterol_total',
       'cholesterol_hdl', 'cholesterol

patient_id                    object
age                            int64
sex                         category
bmi                          float64
enrollment_date       datetime64[ns]
systolic_bp                  float64
diastolic_bp                 float64
cholesterol_total            float64
cholesterol_hdl              float64
cholesterol_ldl              float64
glucose_fasting              float64
site                        category
intervention_group          category
follow_up_months               int64
adverse_events                 int64
outcome_cvd                   object
adherence_pct                float64
dropout                       object
dtype: object

## Part 2: Feature Engineering (8 points)

Create these new calculated columns:

1. `cholesterol_ratio` = cholesterol_ldl / cholesterol_hdl
2. `bp_category` = categorize systolic BP:
   - 'Normal': < 120
   - 'Elevated': 120-129
   - 'High': >= 130
3. `age_group` using `create_bins()` utility:
   - Bins: [0, 40, 55, 70, 100]
   - Labels: ['<40', '40-54', '55-69', '70+']
4. `bmi_category` using standard BMI categories:
   - Underweight: <18.5
   - Normal: 18.5-24.9
   - Overweight: 25-29.9
   - Obese: >=30

In [57]:
# TODO: Calculate cholesterol ratio

df['cholesterol_ratio'] = df["cholesterol_ldl"]/df['cholesterol_hdl']
display(df['cholesterol_ratio'].head())


0    0.745455
1    1.844828
2    1.464286
3    1.857143
4    0.961538
Name: cholesterol_ratio, dtype: float64

In [19]:
# TODO: Categorize blood pressure
bp_category = pd.cut(df['systolic_bp'], bins = [0,119,129,float('inf')], labels= ['Normal','Elevated','High'] )
df['bp_category'] = bp_category
display(df['bp_category'])

0       Elevated
1           High
2       Elevated
3         Normal
4         Normal
          ...   
9995    Elevated
9996    Elevated
9997      Normal
9998        High
9999        High
Name: bp_category, Length: 10000, dtype: category
Categories (3, object): ['Normal' < 'Elevated' < 'High']

**Note:** The `create_bins()` function has an optional `new_column` parameter. If you don't specify it, the new column will be named `{original_column}_binned`. You can use `new_column='age_group'` to give it a custom name.


In [58]:
# TODO: Create age groups
df = create_bins(df,column='age',bins=[0, 40, 55, 70, 100],labels=['<40', '40-54', '55-69', '70+'], new_column = 'age_group')
display(df['age_group'])

0         70+
1         70+
2         70+
3         70+
4         70+
        ...  
9995      70+
9996      70+
9997      70+
9998      70+
9999    55-69
Name: age_group, Length: 10000, dtype: category
Categories (4, object): ['<40' < '40-54' < '55-69' < '70+']

In [28]:
# TODO: Create BMI categories
df = create_bins(df,column='bmi',bins=[0, 18.5, 24.9, 29.9, 100],labels=['<18.5', '18.5-24.9', '25-29.9', '30+'], new_column = 'bmi_category')
display(df['bmi_category'])
print(df.columns)


0         25-29.9
1             NaN
2             NaN
3         25-29.9
4             NaN
          ...    
9995    18.5-24.9
9996      25-29.9
9997    18.5-24.9
9998      25-29.9
9999      25-29.9
Name: bmi_category, Length: 10000, dtype: category
Categories (4, object): ['<18.5' < '18.5-24.9' < '25-29.9' < '30+']

Index(['patient_id', 'age', 'sex', 'bmi', 'enrollment_date', 'systolic_bp',
       'diastolic_bp', 'cholesterol_total', 'cholesterol_hdl',
       'cholesterol_ldl', 'glucose_fasting', 'site', 'intervention_group',
       'follow_up_months', 'adverse_events', 'outcome_cvd', 'adherence_pct',
       'dropout', 'cholesterol_ratio', 'bp_category', 'age_group',
       'bmi_category'],
      dtype='object')


## Part 3: String Cleaning (2 points)

If there are any string columns that need cleaning:
1. Convert to lowercase
2. Strip whitespace
3. Replace any placeholder values

In [59]:
# TODO: String cleaning
display(df.dtypes)
string_columns = df.select_dtypes(include = ['object']).columns
print(string_columns)
placehold = ['unknown', 'na', 'n/a', 'NA', 'missing', 'Missing', '?']
for col in string_columns:
    df[col] = df[col].str.strip().str.lower()
    df[col] = df[col].replace(placehold,pd.NA)


patient_id              object
age                      int64
sex                     object
bmi                    float64
enrollment_date         object
systolic_bp            float64
diastolic_bp           float64
cholesterol_total      float64
cholesterol_hdl        float64
cholesterol_ldl        float64
glucose_fasting        float64
site                    object
intervention_group      object
follow_up_months         int64
adverse_events           int64
outcome_cvd             object
adherence_pct          float64
dropout                 object
cholesterol_ratio      float64
age_group             category
dtype: object

Index(['patient_id', 'sex', 'enrollment_date', 'site', 'intervention_group',
       'outcome_cvd', 'dropout'],
      dtype='object')


## Part 4: One-Hot Encoding (5 points)

Create dummy variables for categorical columns:
1. One-hot encode 'intervention_group' using `pd.get_dummies()`
2. One-hot encode 'site'
3. Drop the original categorical columns
4. Show the new shape and column names

In [72]:
# TODO: One-hot encoding
#df['intervention_group'].unique
dummies = pd.get_dummies(df, columns= ['intervention_group','site'], drop_first=True)
print(dummies)
print(df.columns)
#df = df.drop(['intervention_group'], axis = 1)
print(df.shape)
print(dummies.shape)
print(df.columns)
print(dummies.columns)

     patient_id  age     sex   bmi enrollment_date  systolic_bp  diastolic_bp  \
0        p00001   80       f  29.3      2022-05-01        123.0          80.0   
1        p00002   80  female   NaN      2022-01-06        139.0          81.0   
2        p00003   82  female  -1.0      2023-11-04        123.0          86.0   
3        p00004   95  female  25.4      2022-08-15        116.0          77.0   
4        p00005   95       m   NaN      2023-04-17         97.0          71.0   
...         ...  ...     ...   ...             ...          ...           ...   
9995     p09996   72    male  23.2      2022-04-11        122.0          73.0   
9996     p09997  100  female  28.9      2023-02-10        124.0          78.0   
9997     p09998   78       f  23.8      2023-11-05        110.0          63.0   
9998     p09999   86       f  27.0      2022-08-27        139.0          98.0   
9999     p10000   67  female  29.4      25-03-2022        134.0          83.0   

      cholesterol_total  ch

## Part 5: Save Transformed Data

Save the fully transformed dataset to `output/q6_transformed_data.csv`

In [73]:
# TODO: Save transformed data
# df_transformed.to_csv('output/q6_transformed_data.csv', index=False)
dummies.to_csv('output/q6_transformed_data.csv', index = False)